In [ ]:
import cygnet
import torch, time

test_fixed_precision = False

def total_loss(pred, target):
    return cygnet.focal_loss(pred, target) + cygnet.kl_loss(pred, target)

cygnet.start_timer()

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
torch.set_default_device(device)

net = cygnet.Net()
opt = torch.optim.SGD(net.parameters(), lr=0.001)
scaler = torch.amp.GradScaler("cuda", enabled=True)

b_size = 4
num_batches = 5
epochs = 5

data = [torch.randn(b_size, 1, 2304, 4096) for _ in range(num_batches)]
targets = [torch.randn(b_size, 1, 2304, 4096) for _ in range(num_batches)]

cygnet.end_timer_and_print("setup")
print("\n=======\n")

# fp32
if test_fixed_precision:
    # fp32
    b = 0
    cygnet.start_timer()
    for epoch in range(epochs):
        print(f"start epoch {epoch}:")
        for input, target in zip(data, targets):
            print(f"batch #{b}")
            b += 1
            output = net(input)
            loss = total_loss(output, target)
            loss.backward()
            opt.step()
            opt.zero_grad()
    cygnet.end_timer_and_print("Default precision:")

    dummy = torch.randn(1, 1, 2304, 4096)

    net.eval()
    # Warmup
    for _ in range(10):
        _ = net(dummy)

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        with torch.inference_mode():
            _ = net(dummy)
    torch.cuda.synchronize()
    tim = (time.perf_counter() - t0) / 10 * 1000
    print(f"in inference: {tim:.1f} ms per frame")

    print("\n=======\n")


net = cygnet.Net()
opt = torch.optim.SGD(net.parameters(), lr=0.001)
scaler = torch.amp.GradScaler("cuda", enabled=True)

b = 0
cygnet.start_timer()
for epoch in range(epochs):
    print(f"start epoch {epoch}:")
    for input, target in zip(data, targets):
        print(f"batch #{b}")
        b += 1
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=True):
            output = net(input)
            loss = total_loss(output, target)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()
cygnet.end_timer_and_print("Mixed precision:")

dummy = torch.randn(1, 1, 2304, 4096)

net.eval()
# Warmup
for _ in range(10):
    _ = net(dummy)

torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.float16, enabled=True):
        _ = net(dummy)
torch.cuda.synchronize()
tim = (time.perf_counter() - t0) / 10 * 1000
print(f"in inference: {tim:.1f} ms per frame")

Using cuda device

setup
Total execution time = 0.166 sec
Max memory used by tensors = 2.547 Gbytes


start epoch 0:
batch #0
batch #1
batch #2
batch #3
batch #4
start epoch 1:
batch #5
batch #6
batch #7
batch #8
batch #9
start epoch 2:
batch #10
batch #11
batch #12
batch #13
batch #14
start epoch 3:
batch #15
batch #16
batch #17
batch #18
batch #19
start epoch 4:
batch #20
batch #21
batch #22
batch #23
batch #24

Mixed precision:
Total execution time = 7.872 sec
Max memory used by tensors = 9.056 Gbytes
in inference: 21.0 ms per frame
